# Cold-Start RAG-Enhanced Recommendation Engine
## Spotify-Based Music Recommendation with Semantic Analysis

**Research Objective:** Build a RAG-enhanced recommendation engine that addresses the cold-start problem
in music recommendation by combining retrieval-augmented generation with NLP-based semantic analysis.
Compare performance against traditional Collaborative Filtering (CF) and Content-Based Filtering (CBF) baselines.

---

### Architecture Overview

```
User Query / Seed Track
        |
        v
[ Embedding Layer ] --- sentence-transformers / OpenAI / HuggingFace (compared)
        |
        v
[ FAISS Vector Store ] --- retrieves top-k semantically similar tracks
        |
        v
[ Context Builder ] --- assembles retrieved track metadata into structured context
        |
        v
[ LLM Generation ] --- produces ranked recommendations with explanations
        |
        v
[ Evaluation ] --- compared against CF & CBF baselines
```

### Notebook Roadmap

| Phase | Section | Description |
|-------|---------|-------------|
| 1 | Environment Setup | Dependencies, API keys, project structure |
| 2 | Data Collection | Spotify API ingestion pipeline |
| 3 | EDA & Preprocessing | Feature engineering, text processing |
| 4 | Embedding Generation | Multi-model embedding comparison |
| 5 | FAISS Index | Vector store construction & retrieval |
| 6 | RAG Pipeline | LLM-powered recommendation generation |
| 7 | Baseline Models | Collaborative & Content-Based Filtering |
| 8 | Evaluation Framework | Metrics, comparison, cold-start analysis |
| 9 | API Service | FastAPI wrapper for the recommendation engine |
| 10 | Containerisation | Docker setup for reproducible deployment |

---
## Phase 1: Environment Setup

In [ ]:
# ============================================================
# 1.1 Install Dependencies
# ============================================================
# Run this cell once to install all required packages.
# In production, these go into requirements.txt (see Phase 10).

!pip install spotipy                    # Spotify API wrapper
!pip install pandas numpy scikit-learn  # Core data science
!pip install matplotlib seaborn         # Visualisation
!pip install faiss-cpu                  # FAISS vector store (use faiss-gpu if CUDA available)
!pip install sentence-transformers      # Local embedding models
!pip install openai                     # OpenAI embeddings API
!pip install transformers               # HuggingFace models
!pip install langchain langchain-community langchain-openai  # LLM orchestration
!pip install nltk spacy                 # NLP preprocessing
!pip install implicit                   # ALS-based collaborative filtering
!pip install scipy                      # Sparse matrices for CF
!pip install fastapi uvicorn pydantic   # API serving
!pip install python-dotenv              # Environment variable management
!pip install tqdm                       # Progress bars

# Download spaCy English model
!python -m spacy download en_core_web_sm

In [ ]:
# ============================================================
# 1.2 Project Structure
# ============================================================
# Create the directory layout for the project.
# Keeping data, models, and src separate is essential for
# reproducibility and clean Docker builds.

import os

PROJECT_DIRS = [
    "data/raw",           # Raw Spotify API responses
    "data/processed",     # Cleaned, feature-engineered data
    "data/embeddings",    # Cached embedding vectors
    "data/faiss_index",   # Serialised FAISS indices
    "models",             # Trained baseline models (CF, CBF)
    "src",                # Source modules (refactored from notebook)
    "api",                # FastAPI application code
    "evaluation",         # Evaluation results and plots
    "docker",             # Dockerfile and docker-compose
    "notebooks",          # Additional experiment notebooks
    "config",             # Configuration files
]

for d in PROJECT_DIRS:
    os.makedirs(d, exist_ok=True)
    print(f"Created: {d}/")

print("\nProject structure ready.")

In [ ]:
# ============================================================
# 1.3 Configuration & API Keys
# ============================================================
# IMPORTANT: Never hardcode API keys in notebooks or source code.
# Use a .env file (excluded from git via .gitignore).

# Create a template .env file -- fill in your actual keys
env_template = """# Spotify API credentials
# Register at: https://developer.spotify.com/dashboard
SPOTIFY_CLIENT_ID=your_client_id_here
SPOTIFY_CLIENT_SECRET=your_client_secret_here

# OpenAI API key (for embedding comparison & LLM generation)
# Register at: https://platform.openai.com/api-keys
OPENAI_API_KEY=your_openai_key_here

# Optional: Anthropic API key if using Claude as the LLM
ANTHROPIC_API_KEY=your_anthropic_key_here
"""

with open(".env", "w") as f:
    f.write(env_template)

# Also create .gitignore to prevent leaking credentials
gitignore = """# Secrets
.env

# Data (too large for git -- use DVC or cloud storage)
data/raw/
data/embeddings/
data/faiss_index/
models/

# Python
__pycache__/
*.pyc
.ipynb_checkpoints/
*.egg-info/
venv/
"""

with open(".gitignore", "w") as f:
    f.write(gitignore)

print(".env template and .gitignore created.")
print("ACTION REQUIRED: Fill in your API keys in the .env file before proceeding.")

In [ ]:
# ============================================================
# 1.4 Load Configuration
# ============================================================

from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

load_dotenv()

SPOTIFY_CLIENT_ID = os.getenv("SPOTIFY_CLIENT_ID")
SPOTIFY_CLIENT_SECRET = os.getenv("SPOTIFY_CLIENT_SECRET")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Validate that keys are set
assert SPOTIFY_CLIENT_ID != "your_client_id_here", "Set your Spotify Client ID in .env"
assert SPOTIFY_CLIENT_SECRET != "your_client_secret_here", "Set your Spotify Client Secret in .env"

print("Configuration loaded successfully.")

---
## Phase 2: Spotify API Data Collection

We collect three categories of data from Spotify:

1. **Track metadata** -- name, artist, album, release date, popularity
2. **Audio features** -- danceability, energy, tempo, valence, acousticness, etc.
3. **Artist metadata** -- genres, follower count, popularity

The text fields (track name, artist name, genre tags, album name) will be used
for semantic embedding. The audio features serve as structured features for
the content-based filtering baseline.

**Cold-start relevance:** We deliberately include tracks/artists with low
interaction counts to simulate the cold-start scenario where collaborative
filtering fails but semantic understanding can still work.

In [ ]:
# ============================================================
# 2.1 Spotify API Authentication
# ============================================================

import spotipy
from spotipy.oauth2 import SpotifyClientCredentials

sp = spotipy.Spotify(
    auth_manager=SpotifyClientCredentials(
        client_id=SPOTIFY_CLIENT_ID,
        client_secret=SPOTIFY_CLIENT_SECRET
    ),
    requests_timeout=10,
    retries=3
)

# Quick sanity check
test = sp.search(q="Radiohead", type="artist", limit=1)
print(f"API connection verified. Test query returned: {test['artists']['items'][0]['name']}")

In [ ]:
# ============================================================
# 2.2 Data Collection Pipeline
# ============================================================
# Strategy: Collect tracks from diverse seed genres to ensure
# broad coverage. This also helps create a meaningful cold-start
# test set (niche genres will have fewer user interactions).

import pandas as pd
import time
from tqdm import tqdm

# Seed genres -- choose a mix of popular and niche for cold-start analysis
SEED_GENRES = [
    # Popular (high interaction data)
    "pop", "hip-hop", "rock", "r-n-b", "electronic",
    # Medium popularity
    "jazz", "classical", "country", "reggae", "blues",
    # Niche (cold-start candidates)
    "ambient", "afrobeat", "bossa-nova", "shoegaze", "post-punk",
    "trip-hop", "lo-fi", "drum-and-bass", "neo-soul", "math-rock"
]

TRACKS_PER_GENRE = 50  # Adjust based on your rate limit tolerance


def collect_tracks_by_genre(sp_client, genres, tracks_per_genre=50):
    """
    Collect track data from Spotify by searching within genre categories.
    
    Returns a list of dicts, each representing one track with its
    metadata, audio features, and artist info.
    
    Rate limiting: Spotify allows ~30 requests/minute on the free tier.
    We add a small sleep between batches to stay safe.
    """
    all_tracks = []
    seen_ids = set()  # Deduplicate tracks appearing in multiple genres
    
    for genre in tqdm(genres, desc="Collecting genres"):
        try:
            # Search for tracks in this genre
            # Spotify's search with genre: prefix returns genre-tagged results
            results = sp_client.search(
                q=f"genre:{genre}",
                type="track",
                limit=min(tracks_per_genre, 50),  # Spotify max per request is 50
                market="GB"  # Set your market
            )
            
            tracks = results["tracks"]["items"]
            track_ids = [t["id"] for t in tracks if t["id"] not in seen_ids]
            
            if not track_ids:
                continue
            
            # Fetch audio features in batch (max 100 per request)
            audio_features = sp_client.audio_features(track_ids)
            
            # Build a lookup for audio features
            af_lookup = {}
            for af in audio_features:
                if af:  # Can be None for some tracks
                    af_lookup[af["id"]] = af
            
            for track in tracks:
                tid = track["id"]
                if tid in seen_ids:
                    continue
                seen_ids.add(tid)
                
                af = af_lookup.get(tid, {})
                artist = track["artists"][0]  # Primary artist
                
                # Fetch artist details for genre tags
                # NOTE: This is an extra API call per unique artist.
                # For large-scale collection, batch these or cache them.
                try:
                    artist_info = sp_client.artist(artist["id"])
                    artist_genres = artist_info.get("genres", [])
                    artist_popularity = artist_info.get("popularity", 0)
                    artist_followers = artist_info.get("followers", {}).get("total", 0)
                except Exception:
                    artist_genres = []
                    artist_popularity = 0
                    artist_followers = 0
                
                all_tracks.append({
                    # Identifiers
                    "track_id": tid,
                    "track_name": track["name"],
                    "artist_id": artist["id"],
                    "artist_name": artist["name"],
                    "album_name": track["album"]["name"],
                    "release_date": track["album"].get("release_date", ""),
                    
                    # Popularity (proxy for interaction volume -- key for cold-start)
                    "track_popularity": track["popularity"],
                    "artist_popularity": artist_popularity,
                    "artist_followers": artist_followers,
                    
                    # Genre tags (used for semantic text)
                    "artist_genres": "|".join(artist_genres),
                    "search_genre": genre,
                    
                    # Audio features (structured features for CBF baseline)
                    "danceability": af.get("danceability"),
                    "energy": af.get("energy"),
                    "key": af.get("key"),
                    "loudness": af.get("loudness"),
                    "mode": af.get("mode"),
                    "speechiness": af.get("speechiness"),
                    "acousticness": af.get("acousticness"),
                    "instrumentalness": af.get("instrumentalness"),
                    "liveness": af.get("liveness"),
                    "valence": af.get("valence"),
                    "tempo": af.get("tempo"),
                    "duration_ms": af.get("duration_ms"),
                    "time_signature": af.get("time_signature"),
                })
            
            # Respect rate limits
            time.sleep(1)
            
        except Exception as e:
            print(f"Error collecting genre '{genre}': {e}")
            time.sleep(5)  # Back off on errors
            continue
    
    return all_tracks


# Run collection
print(f"Collecting tracks across {len(SEED_GENRES)} genres...")
raw_tracks = collect_tracks_by_genre(sp, SEED_GENRES, TRACKS_PER_GENRE)

df_raw = pd.DataFrame(raw_tracks)
df_raw.to_csv("data/raw/spotify_tracks_raw.csv", index=False)

print(f"\nCollected {len(df_raw)} unique tracks.")
print(f"Columns: {list(df_raw.columns)}")
df_raw.head()

In [ ]:
# ============================================================
# 2.3 Simulate User Interaction Data (for CF baseline)
# ============================================================
# Since the Spotify API doesn't expose other users' listening
# histories, we simulate interaction data.
#
# WHY THIS IS VALID FOR RESEARCH:
# - The goal is to compare recommendation *methods*, not to
#   build a production system with real user data.
# - We control the simulation to create realistic patterns:
#   popular tracks get more interactions, niche tracks get fewer.
# - This is standard practice in RecSys research when real
#   interaction data isn't available (cite: Rendle et al., 2009).
#
# For your MSc, document this limitation clearly in your methodology.

import numpy as np

np.random.seed(42)

N_USERS = 500
N_TRACKS = len(df_raw)


def simulate_interactions(df, n_users=500, sparsity=0.02):
    """
    Generate a synthetic user-track interaction matrix.
    
    The probability of a user interacting with a track is proportional
    to track popularity, which creates a realistic long-tail distribution.
    
    Args:
        df: DataFrame with a 'track_popularity' column (0-100 scale)
        n_users: Number of synthetic users
        sparsity: Base interaction probability (before popularity weighting)
    
    Returns:
        DataFrame with columns: user_id, track_id, interaction_score
    """
    interactions = []
    
    # Normalise popularity to [0.1, 1.0] range for interaction probability
    pop_scores = df["track_popularity"].values / 100.0
    pop_scores = np.clip(pop_scores, 0.1, 1.0)
    
    for user_id in range(n_users):
        # Each user has genre preferences (random subset)
        # This creates clustering in the interaction matrix
        user_genre_bias = np.random.dirichlet(np.ones(len(SEED_GENRES)) * 0.3)
        genre_to_bias = dict(zip(SEED_GENRES, user_genre_bias))
        
        for idx, row in df.iterrows():
            genre_bias = genre_to_bias.get(row["search_genre"], 0.01)
            interaction_prob = sparsity * pop_scores[idx] * (1 + 10 * genre_bias)
            
            if np.random.random() < interaction_prob:
                # Score: 1-5 scale, biased by popularity
                score = np.clip(
                    np.random.normal(loc=3 + pop_scores[idx], scale=1),
                    1, 5
                )
                interactions.append({
                    "user_id": f"user_{user_id:04d}",
                    "track_id": row["track_id"],
                    "interaction_score": round(score, 1)
                })
    
    return pd.DataFrame(interactions)


df_interactions = simulate_interactions(df_raw, N_USERS)
df_interactions.to_csv("data/raw/user_interactions.csv", index=False)

print(f"Generated {len(df_interactions)} interactions across {N_USERS} users.")
print(f"Sparsity: {1 - len(df_interactions) / (N_USERS * N_TRACKS):.4f}")
print(f"\nInteractions per user (mean): {df_interactions.groupby('user_id').size().mean():.1f}")
print(f"Interactions per track (mean): {df_interactions.groupby('track_id').size().mean():.1f}")
df_interactions.head()

---
## Phase 3: EDA & Preprocessing

Key tasks:
1. Handle missing values in audio features
2. Create the **semantic text field** (combining track name + artist + genres + album)
3. NLP preprocessing on text fields
4. Analyse the popularity distribution (identifies cold-start candidates)
5. Feature normalisation for the CBF baseline

In [ ]:
# ============================================================
# 3.1 Data Cleaning
# ============================================================

df = pd.read_csv("data/raw/spotify_tracks_raw.csv")

print("=== Missing Values ===")
print(df.isnull().sum())
print(f"\n=== Shape: {df.shape}")
print(f"=== Duplicates by track_id: {df['track_id'].duplicated().sum()}")

# Drop rows where audio features are entirely missing
audio_feature_cols = [
    "danceability", "energy", "loudness", "speechiness",
    "acousticness", "instrumentalness", "liveness", "valence", "tempo"
]

df = df.dropna(subset=audio_feature_cols, how="all")

# For partially missing audio features, impute with genre median
for col in audio_feature_cols:
    df[col] = df.groupby("search_genre")[col].transform(
        lambda x: x.fillna(x.median())
    )

# Fill remaining NaNs with global median (in case a genre had all NaN)
df[audio_feature_cols] = df[audio_feature_cols].fillna(df[audio_feature_cols].median())

print(f"\nAfter cleaning: {df.shape[0]} tracks")
print(f"Missing values remaining: {df[audio_feature_cols].isnull().sum().sum()}")

In [ ]:
# ============================================================
# 3.2 Semantic Text Field Construction
# ============================================================
# This is the text that gets embedded for the RAG retrieval layer.
# The richer and more descriptive this text is, the better the
# semantic search quality.
#
# We construct a natural language description of each track that
# combines metadata with audio feature interpretations.

def interpret_audio_features(row):
    """
    Convert numerical audio features into natural language descriptors.
    This bridges the gap between structured data and semantic understanding.
    
    The thresholds are based on Spotify's own documentation for what
    constitutes 'high' vs 'low' for each feature.
    """
    descriptors = []
    
    # Energy
    if row.get("energy", 0.5) > 0.7:
        descriptors.append("high-energy")
    elif row.get("energy", 0.5) < 0.3:
        descriptors.append("calm and mellow")
    
    # Danceability
    if row.get("danceability", 0.5) > 0.7:
        descriptors.append("danceable")
    
    # Valence (musical positiveness)
    if row.get("valence", 0.5) > 0.7:
        descriptors.append("upbeat and cheerful")
    elif row.get("valence", 0.5) < 0.3:
        descriptors.append("melancholic or dark")
    
    # Acousticness
    if row.get("acousticness", 0.5) > 0.7:
        descriptors.append("acoustic")
    
    # Instrumentalness
    if row.get("instrumentalness", 0.0) > 0.5:
        descriptors.append("instrumental")
    
    # Tempo
    tempo = row.get("tempo", 120)
    if tempo and tempo > 140:
        descriptors.append("fast-tempo")
    elif tempo and tempo < 80:
        descriptors.append("slow-tempo")
    
    return ", ".join(descriptors) if descriptors else "moderate"


def build_semantic_text(row):
    """
    Construct a rich text description for embedding.
    
    Format:
    '{track_name}' by {artist_name} from the album '{album_name}'.
    Genre: {genres}. This track is {audio descriptors}.
    """
    genres = row["artist_genres"].replace("|", ", ") if pd.notna(row["artist_genres"]) else "unknown genre"
    audio_desc = interpret_audio_features(row)
    
    text = (
        f"'{row['track_name']}' by {row['artist_name']} "
        f"from the album '{row['album_name']}'. "
        f"Genre: {genres}. "
        f"This track is {audio_desc}."
    )
    return text


df["semantic_text"] = df.apply(build_semantic_text, axis=1)

# Preview
print("=== Sample Semantic Texts ===")
for i, text in enumerate(df["semantic_text"].head(3)):
    print(f"\n[{i+1}] {text}")

In [ ]:
# ============================================================
# 3.3 NLP Preprocessing
# ============================================================
# Light preprocessing for the semantic text.
# We keep it light because transformer-based embeddings work
# best with natural language (not heavily stemmed tokens).

import re
import spacy

nlp = spacy.load("en_core_web_sm")


def preprocess_text(text):
    """
    Light NLP preprocessing:
    - Lowercase
    - Remove special characters (keep alphanumeric, spaces, basic punctuation)
    - Normalise whitespace
    
    We deliberately do NOT stem or lemmatise because:
    1. Transformer embeddings handle morphological variation internally
    2. Genre names like 'shoegaze' or 'trip-hop' would be destroyed
    3. Artist names must remain intact
    """
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s',.-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


df["semantic_text_clean"] = df["semantic_text"].apply(preprocess_text)

print("=== Before vs After Preprocessing ===")
print(f"Original:  {df['semantic_text'].iloc[0][:100]}...")
print(f"Cleaned:   {df['semantic_text_clean'].iloc[0][:100]}...")

In [ ]:
# ============================================================
# 3.4 Cold-Start Analysis (EDA)
# ============================================================
# Identify which tracks qualify as "cold-start" items.
# In our setup, cold-start = low popularity + few simulated interactions.

import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1) Track popularity distribution
axes[0].hist(df["track_popularity"], bins=30, edgecolor="black", alpha=0.7)
axes[0].axvline(x=20, color="red", linestyle="--", label="Cold-start threshold")
axes[0].set_xlabel("Track Popularity")
axes[0].set_ylabel("Count")
axes[0].set_title("Track Popularity Distribution")
axes[0].legend()

# 2) Artist followers (log scale)
axes[1].hist(np.log10(df["artist_followers"].clip(lower=1)), bins=30, edgecolor="black", alpha=0.7)
axes[1].set_xlabel("log10(Artist Followers)")
axes[1].set_ylabel("Count")
axes[1].set_title("Artist Follower Distribution")

# 3) Audio feature correlations
corr = df[audio_feature_cols].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", ax=axes[2], 
            xticklabels=[c[:6] for c in audio_feature_cols],
            yticklabels=[c[:6] for c in audio_feature_cols])
axes[2].set_title("Audio Feature Correlations")

plt.tight_layout()
plt.savefig("evaluation/eda_overview.png", dpi=150, bbox_inches="tight")
plt.show()

# Define cold-start split
COLD_START_THRESHOLD = 20  # tracks with popularity < 20
df["is_cold_start"] = df["track_popularity"] < COLD_START_THRESHOLD

print(f"\nCold-start tracks: {df['is_cold_start'].sum()} ({df['is_cold_start'].mean():.1%})")
print(f"Warm tracks: {(~df['is_cold_start']).sum()} ({(~df['is_cold_start']).mean():.1%})")

In [ ]:
# ============================================================
# 3.5 Feature Normalisation & Save Processed Data
# ============================================================

from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
df[audio_feature_cols] = scaler.fit_transform(df[audio_feature_cols])

# Save processed data
df.to_csv("data/processed/tracks_processed.csv", index=False)

# Save scaler for later use in the API
import pickle
with open("models/audio_feature_scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

print(f"Processed data saved: {df.shape[0]} tracks, {df.shape[1]} columns.")
print(f"Audio features normalised to [0, 1] range.")

---
## Phase 4: Embedding Generation (Multi-Model Comparison)

We generate embeddings using three different approaches and compare them:

1. **sentence-transformers** (`all-MiniLM-L6-v2`) -- lightweight, local, 384-dim
2. **OpenAI** (`text-embedding-3-small`) -- API-based, 1536-dim
3. **HuggingFace** (`BAAI/bge-small-en-v1.5`) -- open-source, 384-dim, strong retrieval perf

Comparison criteria:
- Embedding quality (measured by retrieval relevance in Phase 5)
- Inference speed
- Dimensionality & storage cost
- Cold-start retrieval accuracy

In [ ]:
# ============================================================
# 4.1 Sentence-Transformers Embeddings
# ============================================================

from sentence_transformers import SentenceTransformer
import time

df = pd.read_csv("data/processed/tracks_processed.csv")
texts = df["semantic_text_clean"].tolist()

# --- Model 1: all-MiniLM-L6-v2 ---
print("Loading sentence-transformers model...")
st_model = SentenceTransformer("all-MiniLM-L6-v2")

start = time.time()
embeddings_st = st_model.encode(
    texts,
    show_progress_bar=True,
    batch_size=64,
    normalize_embeddings=True  # L2 normalisation for cosine similarity
)
time_st = time.time() - start

np.save("data/embeddings/embeddings_st_minilm.npy", embeddings_st)

print(f"\nsentence-transformers (all-MiniLM-L6-v2):")
print(f"  Shape: {embeddings_st.shape}")
print(f"  Time: {time_st:.2f}s ({len(texts)/time_st:.0f} texts/sec)")
print(f"  Storage: {embeddings_st.nbytes / 1024 / 1024:.2f} MB")

In [ ]:
# ============================================================
# 4.2 HuggingFace BGE Embeddings
# ============================================================

# --- Model 2: BAAI/bge-small-en-v1.5 ---
# BGE models are specifically trained for retrieval tasks,
# which makes them a strong candidate for the RAG pipeline.
# They expect a specific prefix for queries vs passages.

print("Loading BGE model...")
bge_model = SentenceTransformer("BAAI/bge-small-en-v1.5")

start = time.time()
embeddings_bge = bge_model.encode(
    texts,
    show_progress_bar=True,
    batch_size=64,
    normalize_embeddings=True
)
time_bge = time.time() - start

np.save("data/embeddings/embeddings_bge_small.npy", embeddings_bge)

print(f"\nBGE-small-en-v1.5:")
print(f"  Shape: {embeddings_bge.shape}")
print(f"  Time: {time_bge:.2f}s ({len(texts)/time_bge:.0f} texts/sec)")
print(f"  Storage: {embeddings_bge.nbytes / 1024 / 1024:.2f} MB")

In [ ]:
# ============================================================
# 4.3 OpenAI Embeddings
# ============================================================

from openai import OpenAI

# --- Model 3: text-embedding-3-small ---
# Higher dimensional (1536) but requires API calls.
# Batch limit is 2048 texts per request.

client = OpenAI(api_key=OPENAI_API_KEY)


def get_openai_embeddings(texts, model="text-embedding-3-small", batch_size=100):
    """Embed texts using OpenAI API with batching."""
    all_embeddings = []
    
    for i in tqdm(range(0, len(texts), batch_size), desc="OpenAI embedding"):
        batch = texts[i:i + batch_size]
        response = client.embeddings.create(input=batch, model=model)
        batch_embeds = [item.embedding for item in response.data]
        all_embeddings.extend(batch_embeds)
        time.sleep(0.5)  # Rate limit safety
    
    return np.array(all_embeddings, dtype=np.float32)


start = time.time()
embeddings_oai = get_openai_embeddings(texts)
time_oai = time.time() - start

# L2 normalise for consistency
norms = np.linalg.norm(embeddings_oai, axis=1, keepdims=True)
embeddings_oai = embeddings_oai / norms

np.save("data/embeddings/embeddings_openai_small.npy", embeddings_oai)

print(f"\nOpenAI text-embedding-3-small:")
print(f"  Shape: {embeddings_oai.shape}")
print(f"  Time: {time_oai:.2f}s ({len(texts)/time_oai:.0f} texts/sec)")
print(f"  Storage: {embeddings_oai.nbytes / 1024 / 1024:.2f} MB")

In [ ]:
# ============================================================
# 4.4 Embedding Comparison Summary
# ============================================================

comparison = pd.DataFrame({
    "Model": ["MiniLM-L6-v2", "BGE-small-en", "OpenAI-3-small"],
    "Dimensions": [embeddings_st.shape[1], embeddings_bge.shape[1], embeddings_oai.shape[1]],
    "Encoding Time (s)": [round(time_st, 2), round(time_bge, 2), round(time_oai, 2)],
    "Throughput (texts/s)": [
        round(len(texts)/time_st), 
        round(len(texts)/time_bge), 
        round(len(texts)/time_oai)
    ],
    "Storage (MB)": [
        round(embeddings_st.nbytes/1024/1024, 2),
        round(embeddings_bge.nbytes/1024/1024, 2),
        round(embeddings_oai.nbytes/1024/1024, 2)
    ],
    "Local/API": ["Local", "Local", "API"],
})

print("=== Embedding Model Comparison ===")
print(comparison.to_string(index=False))
print("\n(Retrieval quality comparison will be done in Phase 5 after FAISS indexing)")

---
## Phase 5: FAISS Index Construction & Retrieval

Build a FAISS index for each embedding model. FAISS (Facebook AI Similarity Search)
supports multiple index types -- we use `IndexFlatIP` (inner product on normalised
vectors = cosine similarity) for exact search, then compare with `IndexIVFFlat`
for approximate search at scale.

In [ ]:
# ============================================================
# 5.1 Build FAISS Indices
# ============================================================

import faiss


def build_faiss_index(embeddings, index_type="flat", nlist=50):
    """
    Build a FAISS index from embedding vectors.
    
    Args:
        embeddings: numpy array of shape (n_items, dim)
        index_type: 'flat' for exact search, 'ivf' for approximate
        nlist: number of Voronoi cells for IVF index
    
    Returns:
        FAISS index object
    """
    dim = embeddings.shape[1]
    
    if index_type == "flat":
        # Exact search using inner product (cosine sim on L2-normalised vectors)
        index = faiss.IndexFlatIP(dim)
    elif index_type == "ivf":
        # Approximate search -- faster for large datasets
        quantiser = faiss.IndexFlatIP(dim)
        index = faiss.IndexIVFFlat(quantiser, dim, nlist, faiss.METRIC_INNER_PRODUCT)
        index.train(embeddings)  # IVF requires training
        index.nprobe = 10  # Number of cells to search (trade-off: speed vs recall)
    
    index.add(embeddings)
    return index


# Build indices for each embedding model
embedding_files = {
    "MiniLM": "data/embeddings/embeddings_st_minilm.npy",
    "BGE": "data/embeddings/embeddings_bge_small.npy",
    "OpenAI": "data/embeddings/embeddings_openai_small.npy",
}

indices = {}
for name, path in embedding_files.items():
    embeds = np.load(path)
    idx = build_faiss_index(embeds, index_type="flat")
    faiss.write_index(idx, f"data/faiss_index/index_{name.lower()}.faiss")
    indices[name] = idx
    print(f"Built FAISS index for {name}: {idx.ntotal} vectors, dim={embeds.shape[1]}")

print("\nAll FAISS indices built and saved.")

In [ ]:
# ============================================================
# 5.2 Retrieval Quality Test
# ============================================================
# Test retrieval with a sample query across all three embedding models.
# This gives us a qualitative sense of retrieval quality before
# running formal evaluation.

def retrieve_similar(query_text, model, index, df, top_k=5):
    """
    Retrieve top-k similar tracks for a text query.
    
    Args:
        query_text: natural language query
        model: embedding model (must have .encode() method)
        index: FAISS index
        df: DataFrame with track metadata
        top_k: number of results
    
    Returns:
        DataFrame of top-k results with similarity scores
    """
    query_embedding = model.encode(
        [query_text], normalize_embeddings=True
    ).astype(np.float32)
    
    scores, ids = index.search(query_embedding, top_k)
    
    results = df.iloc[ids[0]][["track_name", "artist_name", "artist_genres", "track_popularity"]].copy()
    results["similarity_score"] = scores[0]
    return results


# Test query
test_query = "upbeat electronic dance music with high energy"

print(f"Query: '{test_query}'\n")

for name, model in [("MiniLM", st_model), ("BGE", bge_model)]:
    print(f"\n--- {name} Results ---")
    results = retrieve_similar(test_query, model, indices[name], df)
    for _, row in results.iterrows():
        print(f"  [{row['similarity_score']:.3f}] {row['track_name']} - {row['artist_name']}")

# For OpenAI, we need to use the API for the query embedding
print(f"\n--- OpenAI Results ---")
query_resp = client.embeddings.create(input=[test_query], model="text-embedding-3-small")
query_embed_oai = np.array([query_resp.data[0].embedding], dtype=np.float32)
query_embed_oai = query_embed_oai / np.linalg.norm(query_embed_oai)

scores, ids = indices["OpenAI"].search(query_embed_oai, 5)
for i, (score, idx) in enumerate(zip(scores[0], ids[0])):
    row = df.iloc[idx]
    print(f"  [{score:.3f}] {row['track_name']} - {row['artist_name']}")

---
## Phase 6: RAG Pipeline (LLM-Powered Recommendations)

This is the core of the research contribution. The RAG pipeline:

1. Takes a user query (or seed track)
2. Retrieves top-k similar tracks from FAISS
3. Assembles the retrieved tracks into a structured context
4. Feeds the context + query to an LLM
5. The LLM generates ranked recommendations with explanations

**Cold-start advantage:** Unlike CF, this pipeline can recommend tracks
that have zero interaction history, as long as they have descriptive metadata.

In [ ]:
# ============================================================
# 6.1 Context Builder
# ============================================================

def build_rag_context(query, model, index, df, top_k=10):
    """
    Retrieve tracks and format them as structured context for the LLM.
    
    The context format is deliberately structured to make it easy
    for the LLM to reason about track similarity and make comparisons.
    
    Args:
        query: user's natural language query
        model: embedding model
        index: FAISS index
        df: processed tracks DataFrame
        top_k: number of tracks to retrieve for context
    
    Returns:
        Formatted context string for the LLM prompt
    """
    # Encode and retrieve
    query_embedding = model.encode(
        [query], normalize_embeddings=True
    ).astype(np.float32)
    
    scores, ids = index.search(query_embedding, top_k)
    
    # Build context
    context_parts = []
    for rank, (score, idx) in enumerate(zip(scores[0], ids[0]), 1):
        track = df.iloc[idx]
        context_parts.append(
            f"[Track {rank}] (Similarity: {score:.3f})\n"
            f"  Title: {track['track_name']}\n"
            f"  Artist: {track['artist_name']}\n"
            f"  Album: {track['album_name']}\n"
            f"  Genres: {track.get('artist_genres', 'N/A')}\n"
            f"  Popularity: {track['track_popularity']}/100\n"
            f"  Energy: {track.get('energy', 'N/A'):.2f}, "
            f"Danceability: {track.get('danceability', 'N/A'):.2f}, "
            f"Valence: {track.get('valence', 'N/A'):.2f}\n"
            f"  Tempo: {track.get('tempo', 'N/A'):.0f} BPM"
        )
    
    return "\n\n".join(context_parts)


# Preview context for a test query
test_context = build_rag_context(
    "I want something similar to Radiohead but more electronic",
    bge_model, indices["BGE"], df, top_k=5
)
print(test_context)

In [ ]:
# ============================================================
# 6.2 LLM Recommendation Generator
# ============================================================

from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate


# System prompt for the recommendation LLM
SYSTEM_PROMPT = """You are a music recommendation engine. You receive a user's
query and a set of candidate tracks retrieved from a music database based on
semantic similarity.

Your job is to:
1. Analyse the user's query to understand their intent, mood, and preferences.
2. Review the candidate tracks and their attributes.
3. Select and rank the top 5 most relevant recommendations.
4. For each recommendation, provide a brief explanation of why it matches the query.

Respond in the following JSON format only:
{
  "recommendations": [
    {
      "rank": 1,
      "track_name": "...",
      "artist_name": "...",
      "explanation": "...",
      "confidence": 0.0-1.0
    }
  ],
  "query_interpretation": "Brief summary of what you understood from the query"
}
"""

USER_PROMPT_TEMPLATE = """User query: {query}

Retrieved candidate tracks:
{context}

Based on the above candidates, provide your top 5 ranked recommendations.
"""


class RAGRecommender:
    """
    End-to-end RAG-based recommendation engine.
    
    Combines FAISS retrieval with LLM-powered ranking and explanation.
    """
    
    def __init__(self, embedding_model, faiss_index, track_df, llm_model="gpt-4o-mini"):
        self.embedding_model = embedding_model
        self.faiss_index = faiss_index
        self.track_df = track_df
        self.llm = ChatOpenAI(model=llm_model, temperature=0.3)
        self.prompt = ChatPromptTemplate.from_messages([
            ("system", SYSTEM_PROMPT),
            ("user", USER_PROMPT_TEMPLATE)
        ])
        self.chain = self.prompt | self.llm
    
    def recommend(self, query, top_k_retrieve=10, top_k_recommend=5):
        """
        Generate recommendations for a user query.
        
        Args:
            query: natural language query
            top_k_retrieve: number of candidates to retrieve from FAISS
            top_k_recommend: number of final recommendations from LLM
        
        Returns:
            dict with recommendations and metadata
        """
        # Step 1: Retrieve candidates
        context = build_rag_context(
            query, self.embedding_model, self.faiss_index,
            self.track_df, top_k=top_k_retrieve
        )
        
        # Step 2: LLM generation
        response = self.chain.invoke({
            "query": query,
            "context": context
        })
        
        # Step 3: Parse response
        import json
        try:
            result = json.loads(response.content)
        except json.JSONDecodeError:
            # Fallback: try to extract JSON from the response
            content = response.content
            json_start = content.find("{")
            json_end = content.rfind("}") + 1
            result = json.loads(content[json_start:json_end])
        
        result["metadata"] = {
            "query": query,
            "candidates_retrieved": top_k_retrieve,
            "embedding_model": type(self.embedding_model).__name__
        }
        
        return result


# Instantiate recommender with BGE embeddings (you can swap models here)
rag_recommender = RAGRecommender(
    embedding_model=bge_model,
    faiss_index=indices["BGE"],
    track_df=df
)

print("RAG Recommender initialised.")

In [ ]:
# ============================================================
# 6.3 Test the RAG Pipeline
# ============================================================

import json

test_queries = [
    "I want upbeat pop songs for a workout playlist",
    "Recommend me something like Radiohead but more electronic",
    "Calm acoustic music for studying late at night",
    "Dark, atmospheric trip-hop similar to Massive Attack",
]

for query in test_queries:
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print(f"{'='*60}")
    
    result = rag_recommender.recommend(query)
    
    print(f"\nInterpretation: {result.get('query_interpretation', 'N/A')}")
    print(f"\nRecommendations:")
    for rec in result["recommendations"]:
        print(f"  {rec['rank']}. {rec['track_name']} - {rec['artist_name']}")
        print(f"     {rec['explanation']}")
        print(f"     Confidence: {rec['confidence']}")

---
## Phase 7: Baseline Models

We implement two baselines that the RAG approach will be compared against:

1. **Collaborative Filtering (CF)** -- ALS matrix factorisation on user-track interactions
2. **Content-Based Filtering (CBF)** -- cosine similarity on audio features

Both baselines are standard in RecSys literature and represent the state of practice
that RAG-enhanced approaches aim to improve upon, especially in cold-start scenarios.

In [ ]:
# ============================================================
# 7.1 Collaborative Filtering (ALS)
# ============================================================
# Using the `implicit` library which implements ALS (Alternating
# Least Squares) for implicit feedback matrix factorisation.
#
# KNOWN LIMITATION: CF will fail on cold-start items (tracks with
# zero or very few interactions). This is by design -- it's the
# weakness our RAG approach aims to address.

from scipy.sparse import csr_matrix
from implicit.als import AlternatingLeastSquares

# Load interaction data
df_interactions = pd.read_csv("data/raw/user_interactions.csv")

# Create mappings
user_ids = df_interactions["user_id"].unique()
track_ids = df_interactions["track_id"].unique()

user_to_idx = {uid: i for i, uid in enumerate(user_ids)}
track_to_idx = {tid: i for i, tid in enumerate(track_ids)}
idx_to_track = {i: tid for tid, i in track_to_idx.items()}

# Build sparse interaction matrix
rows = df_interactions["user_id"].map(user_to_idx).values
cols = df_interactions["track_id"].map(track_to_idx).values
vals = df_interactions["interaction_score"].values

interaction_matrix = csr_matrix(
    (vals, (rows, cols)),
    shape=(len(user_ids), len(track_ids))
)

print(f"Interaction matrix: {interaction_matrix.shape}")
print(f"Sparsity: {1 - interaction_matrix.nnz / (interaction_matrix.shape[0] * interaction_matrix.shape[1]):.4f}")

# Train ALS model
als_model = AlternatingLeastSquares(
    factors=64,
    regularization=0.1,
    iterations=20,
    random_state=42
)

als_model.fit(interaction_matrix)
print("\nALS model trained.")

# Save model
with open("models/als_model.pkl", "wb") as f:
    pickle.dump({
        "model": als_model,
        "user_to_idx": user_to_idx,
        "track_to_idx": track_to_idx,
        "idx_to_track": idx_to_track,
    }, f)

In [ ]:
# ============================================================
# 7.2 CF Recommendation Function
# ============================================================

def cf_recommend(user_id, als_model, interaction_matrix, user_to_idx, idx_to_track, df, top_k=5):
    """
    Generate recommendations using Collaborative Filtering.
    
    Args:
        user_id: string user identifier
        als_model: trained ALS model
        interaction_matrix: sparse user-track matrix
        user_to_idx: user ID to index mapping
        idx_to_track: index to track ID mapping
        df: track metadata DataFrame
        top_k: number of recommendations
    
    Returns:
        DataFrame of recommendations
    """
    if user_id not in user_to_idx:
        return pd.DataFrame()  # Cold-start user: CF cannot recommend
    
    user_idx = user_to_idx[user_id]
    
    # Get recommendations (items the user hasn't interacted with)
    track_indices, scores = als_model.recommend(
        user_idx,
        interaction_matrix[user_idx],
        N=top_k,
        filter_already_liked_items=True
    )
    
    rec_track_ids = [idx_to_track[i] for i in track_indices]
    results = df[df["track_id"].isin(rec_track_ids)][["track_name", "artist_name", "track_popularity"]].copy()
    results["cf_score"] = scores[:len(results)]
    
    return results


# Test CF
test_user = user_ids[0]
cf_results = cf_recommend(test_user, als_model, interaction_matrix, user_to_idx, idx_to_track, df)
print(f"CF recommendations for {test_user}:")
print(cf_results.to_string(index=False))

In [ ]:
# ============================================================
# 7.3 Content-Based Filtering
# ============================================================
# Pure audio-feature similarity using cosine distance.
# This is a "dumb" baseline -- no NLP, no user history.

from sklearn.metrics.pairwise import cosine_similarity


def cbf_recommend(seed_track_id, df, audio_feature_cols, top_k=5):
    """
    Content-Based Filtering: recommend tracks similar to a seed track
    based on audio features only.
    
    Args:
        seed_track_id: Spotify track ID of the seed
        df: processed DataFrame
        audio_feature_cols: list of audio feature column names
        top_k: number of recommendations
    
    Returns:
        DataFrame of recommendations with similarity scores
    """
    seed_idx = df[df["track_id"] == seed_track_id].index
    if len(seed_idx) == 0:
        return pd.DataFrame()  # Track not in dataset
    
    seed_idx = seed_idx[0]
    seed_features = df.loc[seed_idx, audio_feature_cols].values.reshape(1, -1)
    all_features = df[audio_feature_cols].values
    
    similarities = cosine_similarity(seed_features, all_features)[0]
    
    # Exclude the seed track itself
    similarities[seed_idx] = -1
    
    top_indices = np.argsort(similarities)[::-1][:top_k]
    
    results = df.iloc[top_indices][["track_name", "artist_name", "track_popularity"]].copy()
    results["cbf_similarity"] = similarities[top_indices]
    
    return results


# Test CBF
seed_track = df.iloc[0]["track_id"]
print(f"CBF recommendations for seed: '{df.iloc[0]['track_name']}' by {df.iloc[0]['artist_name']}")
cbf_results = cbf_recommend(seed_track, df, audio_feature_cols)
print(cbf_results.to_string(index=False))

---
## Phase 8: Evaluation Framework

### Metrics

We evaluate all three approaches (RAG, CF, CBF) using standard RecSys metrics:

1. **Precision@K** -- fraction of recommended items that are relevant
2. **Recall@K** -- fraction of relevant items that are recommended
3. **NDCG@K** -- normalised discounted cumulative gain (ranking quality)
4. **Hit Rate@K** -- fraction of users for whom at least one rec is relevant
5. **Coverage** -- fraction of total catalogue recommended across all users

### Cold-Start Specific Evaluation

We split evaluation into:
- **Warm items** (popularity >= 20): all methods should perform reasonably
- **Cold-start items** (popularity < 20): CF is expected to fail; RAG should excel

In [ ]:
# ============================================================
# 8.1 Evaluation Metrics
# ============================================================

from sklearn.metrics import ndcg_score


def precision_at_k(recommended, relevant, k):
    """Fraction of top-k recommendations that are relevant."""
    rec_set = set(recommended[:k])
    rel_set = set(relevant)
    return len(rec_set & rel_set) / k if k > 0 else 0.0


def recall_at_k(recommended, relevant, k):
    """Fraction of relevant items that appear in top-k."""
    rec_set = set(recommended[:k])
    rel_set = set(relevant)
    return len(rec_set & rel_set) / len(rel_set) if len(rel_set) > 0 else 0.0


def hit_rate_at_k(recommended, relevant, k):
    """1 if at least one relevant item is in top-k, else 0."""
    rec_set = set(recommended[:k])
    rel_set = set(relevant)
    return 1.0 if len(rec_set & rel_set) > 0 else 0.0


def catalogue_coverage(all_recommendations, total_items):
    """Fraction of total catalogue that appears in any recommendation."""
    unique_recs = set()
    for recs in all_recommendations:
        unique_recs.update(recs)
    return len(unique_recs) / total_items if total_items > 0 else 0.0


print("Evaluation metrics defined.")

In [ ]:
# ============================================================
# 8.2 Train/Test Split for Evaluation
# ============================================================
# For each user, hold out 20% of their interactions as test set.
# The remaining 80% is used for training CF and as ground truth
# for evaluation.

from sklearn.model_selection import GroupShuffleSplit

# For each user, split interactions into train and test
train_interactions = []
test_interactions = []

for user_id, group in df_interactions.groupby("user_id"):
    if len(group) < 5:
        # Users with very few interactions go entirely to train
        train_interactions.append(group)
        continue
    
    n_test = max(1, int(len(group) * 0.2))
    test_sample = group.sample(n=n_test, random_state=42)
    train_sample = group.drop(test_sample.index)
    
    train_interactions.append(train_sample)
    test_interactions.append(test_sample)

df_train = pd.concat(train_interactions, ignore_index=True)
df_test = pd.concat(test_interactions, ignore_index=True)

print(f"Train interactions: {len(df_train)}")
print(f"Test interactions: {len(df_test)}")
print(f"Test users: {df_test['user_id'].nunique()}")

In [ ]:
# ============================================================
# 8.3 Run Evaluation
# ============================================================
# This cell runs evaluation for all three methods.
# NOTE: The RAG evaluation is expensive (LLM API calls per user).
# For initial development, evaluate on a subset of users.

K = 5  # Evaluate at k=5
N_EVAL_USERS = 50  # Subset for RAG evaluation (adjust based on budget)

eval_users = df_test["user_id"].unique()[:N_EVAL_USERS]

results = {"CF": [], "CBF": [], "RAG": []}

for user_id in tqdm(eval_users, desc="Evaluating"):
    # Ground truth: tracks the user actually interacted with in test set
    user_test = df_test[df_test["user_id"] == user_id]
    relevant_tracks = set(user_test["track_id"].tolist())
    
    if len(relevant_tracks) == 0:
        continue
    
    # --- CF Evaluation ---
    try:
        cf_recs = cf_recommend(user_id, als_model, interaction_matrix, user_to_idx, idx_to_track, df, top_k=K)
        cf_track_ids = df[df["track_name"].isin(cf_recs["track_name"])]["track_id"].tolist()[:K]
    except Exception:
        cf_track_ids = []
    
    results["CF"].append({
        "user_id": user_id,
        "precision": precision_at_k(cf_track_ids, relevant_tracks, K),
        "recall": recall_at_k(cf_track_ids, relevant_tracks, K),
        "hit_rate": hit_rate_at_k(cf_track_ids, relevant_tracks, K),
    })
    
    # --- CBF Evaluation ---
    # Use the user's most-interacted track as seed
    user_train = df_train[df_train["user_id"] == user_id]
    if len(user_train) > 0:
        seed_track = user_train.sort_values("interaction_score", ascending=False).iloc[0]["track_id"]
        cbf_recs = cbf_recommend(seed_track, df, audio_feature_cols, top_k=K)
        cbf_track_ids = df[df["track_name"].isin(cbf_recs["track_name"])]["track_id"].tolist()[:K]
    else:
        cbf_track_ids = []
    
    results["CBF"].append({
        "user_id": user_id,
        "precision": precision_at_k(cbf_track_ids, relevant_tracks, K),
        "recall": recall_at_k(cbf_track_ids, relevant_tracks, K),
        "hit_rate": hit_rate_at_k(cbf_track_ids, relevant_tracks, K),
    })
    
    # --- RAG Evaluation ---
    # Generate a query from the user's listening history
    user_tracks_info = df[df["track_id"].isin(user_train["track_id"])]
    if len(user_tracks_info) > 0:
        top_genres = user_tracks_info["artist_genres"].str.split("|").explode().value_counts().head(3).index.tolist()
        query = f"Recommend music similar to {top_genres[0] if top_genres else 'pop'} genre"
        
        try:
            rag_result = rag_recommender.recommend(query, top_k_retrieve=10, top_k_recommend=K)
            rag_track_names = [r["track_name"] for r in rag_result.get("recommendations", [])]
            rag_track_ids = df[df["track_name"].isin(rag_track_names)]["track_id"].tolist()[:K]
        except Exception as e:
            rag_track_ids = []
    else:
        rag_track_ids = []
    
    results["RAG"].append({
        "user_id": user_id,
        "precision": precision_at_k(rag_track_ids, relevant_tracks, K),
        "recall": recall_at_k(rag_track_ids, relevant_tracks, K),
        "hit_rate": hit_rate_at_k(rag_track_ids, relevant_tracks, K),
    })

print("Evaluation complete.")

In [ ]:
# ============================================================
# 8.4 Results Summary & Visualisation
# ============================================================

summary = {}
for method, data in results.items():
    df_results = pd.DataFrame(data)
    summary[method] = {
        "Precision@5": df_results["precision"].mean(),
        "Recall@5": df_results["recall"].mean(),
        "Hit Rate@5": df_results["hit_rate"].mean(),
    }

df_summary = pd.DataFrame(summary).T
print("=== Overall Results ===")
print(df_summary.round(4).to_string())

# Visualisation
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, metric in enumerate(["Precision@5", "Recall@5", "Hit Rate@5"]):
    values = [summary[m][metric] for m in ["CF", "CBF", "RAG"]]
    bars = axes[i].bar(["CF", "CBF", "RAG"], values, color=["#2196F3", "#FF9800", "#4CAF50"])
    axes[i].set_title(metric)
    axes[i].set_ylim(0, max(values) * 1.3 if max(values) > 0 else 1)
    for bar, val in zip(bars, values):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f"{val:.3f}", ha="center", fontsize=10)

plt.suptitle(f"Recommendation Method Comparison (K={K}, N={N_EVAL_USERS} users)", fontsize=14)
plt.tight_layout()
plt.savefig("evaluation/method_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# 8.5 Cold-Start Specific Analysis
# ============================================================
# This is the key research finding: how does each method perform
# when the test items are cold-start (low popularity) vs warm?

# TODO: After running full evaluation, split results by whether
# the relevant test tracks are cold-start or warm.
# This analysis should show:
#   - CF performance drops dramatically for cold-start items
#   - CBF maintains some performance (feature-based, no interaction needed)
#   - RAG maintains or improves (semantic understanding of metadata)

print("""COLD-START ANALYSIS TEMPLATE
==============================
After full evaluation, implement:

1. Split test set by cold-start flag
2. Recompute metrics separately for warm vs cold-start items
3. Statistical significance test (paired t-test or Wilcoxon)
4. Generate warm vs cold-start comparison chart

Expected findings:
- CF: strong on warm items, near-zero on cold-start
- CBF: moderate on both (feature similarity doesn't need interactions)
- RAG: strong on both (semantic understanding + LLM reasoning)
""")

---
## Phase 9: FastAPI Service

Wrap the recommendation engine in a REST API for serving predictions.
This is the interface that gets containerised in Phase 10.

In [ ]:
# ============================================================
# 9.1 Write the FastAPI Application
# ============================================================
# This writes the API code to a file. You would normally develop
# this outside the notebook, but having it here keeps the project
# self-contained for the starter template.

api_code = '''
"""FastAPI service for RAG-enhanced music recommendations."""

import os
import json
import pickle
import numpy as np
import pandas as pd
import faiss
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from sentence_transformers import SentenceTransformer
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from dotenv import load_dotenv

load_dotenv()

# ---- Pydantic Models ----

class RecommendationRequest(BaseModel):
    query: str = Field(..., description="Natural language query for recommendations")
    top_k: int = Field(default=5, ge=1, le=20, description="Number of recommendations")
    embedding_model: str = Field(default="bge", description="Embedding model: bge, minilm, openai")


class TrackRecommendation(BaseModel):
    rank: int
    track_name: str
    artist_name: str
    explanation: str
    confidence: float


class RecommendationResponse(BaseModel):
    recommendations: list[TrackRecommendation]
    query_interpretation: str
    embedding_model_used: str


# ---- App Initialisation ----

app = FastAPI(
    title="RAG Music Recommender",
    description="Cold-start RAG-enhanced music recommendation API",
    version="0.1.0"
)

# Global state (loaded once at startup)
state = {}


@app.on_event("startup")
def load_models():
    """Load all models and data at startup."""
    print("Loading models...")
    
    # Track data
    state["df"] = pd.read_csv("data/processed/tracks_processed.csv")
    
    # Embedding models
    state["bge_model"] = SentenceTransformer("BAAI/bge-small-en-v1.5")
    state["minilm_model"] = SentenceTransformer("all-MiniLM-L6-v2")
    
    # FAISS indices
    state["bge_index"] = faiss.read_index("data/faiss_index/index_bge.faiss")
    state["minilm_index"] = faiss.read_index("data/faiss_index/index_minilm.faiss")
    
    # LLM
    state["llm"] = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)
    
    print("All models loaded.")


SYSTEM_PROMPT = """You are a music recommendation engine. Given a user query and
candidate tracks, select and rank the top recommendations. Respond in JSON only:
{"recommendations": [{"rank": 1, "track_name": "...", "artist_name": "...",
"explanation": "...", "confidence": 0.0-1.0}], "query_interpretation": "..."}"""


@app.post("/recommend", response_model=RecommendationResponse)
def recommend(request: RecommendationRequest):
    """Generate music recommendations using RAG pipeline."""
    
    # Select embedding model and index
    model_map = {
        "bge": (state["bge_model"], state["bge_index"]),
        "minilm": (state["minilm_model"], state["minilm_index"]),
    }
    
    if request.embedding_model not in model_map:
        raise HTTPException(status_code=400, detail=f"Unknown model: {request.embedding_model}")
    
    embed_model, faiss_index = model_map[request.embedding_model]
    df = state["df"]
    
    # Retrieve candidates
    query_embedding = embed_model.encode(
        [request.query], normalize_embeddings=True
    ).astype(np.float32)
    
    scores, ids = faiss_index.search(query_embedding, request.top_k * 2)
    
    # Build context
    context_parts = []
    for rank, (score, idx) in enumerate(zip(scores[0], ids[0]), 1):
        track = df.iloc[idx]
        context_parts.append(
            f"[{rank}] {track[\"track_name\"]} by {track[\"artist_name\"]} "
            f"(Genres: {track.get(\"artist_genres\", \"N/A\")}, "
            f"Similarity: {score:.3f})"
        )
    
    context = "\\n".join(context_parts)
    
    # LLM generation
    prompt = ChatPromptTemplate.from_messages([
        ("system", SYSTEM_PROMPT),
        ("user", f"Query: {request.query}\\n\\nCandidates:\\n{context}")
    ])
    
    chain = prompt | state["llm"]
    response = chain.invoke({})
    
    # Parse
    try:
        result = json.loads(response.content)
    except json.JSONDecodeError:
        content = response.content
        result = json.loads(content[content.find("{"):content.rfind("}")+1])
    
    return RecommendationResponse(
        recommendations=[TrackRecommendation(**r) for r in result["recommendations"][:request.top_k]],
        query_interpretation=result.get("query_interpretation", ""),
        embedding_model_used=request.embedding_model
    )


@app.get("/health")
def health():
    return {"status": "healthy", "tracks_loaded": len(state.get("df", []))}
'''

with open("api/main.py", "w") as f:
    f.write(api_code)

print("FastAPI application written to api/main.py")
print("\nTo run locally:")
print("  cd <project_root>")
print("  uvicorn api.main:app --reload --port 8000")
print("  Open http://localhost:8000/docs for interactive API docs")

---
## Phase 10: Docker Containerisation

Package the entire application into a Docker container for
reproducible deployment. The container includes the API server,
all models, and data.

In [ ]:
# ============================================================
# 10.1 Generate Dockerfile
# ============================================================

dockerfile = '''# ============================================================
# RAG Music Recommender -- Production Docker Image
# ============================================================
# Multi-stage build:
#   Stage 1: Install Python dependencies
#   Stage 2: Copy app code and data, run server
#
# Build:  docker build -t rag-recsys .
# Run:    docker run -p 8000:8000 --env-file .env rag-recsys

# --- Stage 1: Dependencies ---
FROM python:3.11-slim AS builder

WORKDIR /install

# System deps for FAISS and scientific packages
RUN apt-get update && apt-get install -y --no-install-recommends \\
    build-essential \\
    && rm -rf /var/lib/apt/lists/*

COPY requirements.txt .
RUN pip install --no-cache-dir --prefix=/install/deps -r requirements.txt

# --- Stage 2: Application ---
FROM python:3.11-slim

WORKDIR /app

# Copy installed packages from builder
COPY --from=builder /install/deps /usr/local

# Copy application code
COPY api/ ./api/
COPY data/processed/ ./data/processed/
COPY data/faiss_index/ ./data/faiss_index/
COPY models/ ./models/
COPY config/ ./config/

# Pre-download embedding models at build time (avoids runtime download)
RUN python -c "from sentence_transformers import SentenceTransformer; \\
    SentenceTransformer('BAAI/bge-small-en-v1.5'); \\
    SentenceTransformer('all-MiniLM-L6-v2')"

# Expose API port
EXPOSE 8000

# Health check
HEALTHCHECK --interval=30s --timeout=10s --start-period=60s --retries=3 \\
    CMD python -c "import urllib.request; urllib.request.urlopen('http://localhost:8000/health')"

# Run the API server
# Workers: 1 is fine for GPU-bound inference; scale horizontally with k8s
CMD ["uvicorn", "api.main:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "1"]
'''

with open("Dockerfile", "w") as f:
    f.write(dockerfile)

print("Dockerfile written.")

In [ ]:
# ============================================================
# 10.2 Generate requirements.txt
# ============================================================

requirements = """# Core
fastapi==0.109.0
uvicorn[standard]==0.27.0
pydantic==2.5.3
python-dotenv==1.0.0

# Data
pandas==2.1.4
numpy==1.26.3
scikit-learn==1.4.0
scipy==1.12.0

# Embeddings & Vector Store
faiss-cpu==1.7.4
sentence-transformers==2.3.1
transformers==4.37.0
torch==2.1.2

# LLM
openai==1.10.0
langchain==0.1.4
langchain-community==0.0.14
langchain-openai==0.0.5

# NLP
spacy==3.7.2
nltk==3.8.1

# Collaborative Filtering
implicit==0.7.2

# Data Collection
spotipy==2.23.0
"""

with open("requirements.txt", "w") as f:
    f.write(requirements)

print("requirements.txt written.")

In [ ]:
# ============================================================
# 10.3 Generate docker-compose.yml
# ============================================================
# docker-compose makes it easy to add services later
# (e.g., a Redis cache, a monitoring stack, a frontend)

compose = """version: '3.8'

services:
  recommender:
    build:
      context: .
      dockerfile: Dockerfile
    ports:
      - "8000:8000"
    env_file:
      - .env
    volumes:
      # Mount data directory for easy updates without rebuilding
      - ./data/processed:/app/data/processed:ro
      - ./data/faiss_index:/app/data/faiss_index:ro
    restart: unless-stopped
    deploy:
      resources:
        limits:
          memory: 4G
        reservations:
          memory: 2G
"""

with open("docker-compose.yml", "w") as f:
    f.write(compose)

print("docker-compose.yml written.")
print("\nTo build and run:")
print("  docker-compose up --build")
print("\nTo run in detached mode:")
print("  docker-compose up -d --build")

In [ ]:
# ============================================================
# 10.4 Generate .dockerignore
# ============================================================

dockerignore = """# Don't send unnecessary files to Docker build context
.git
.gitignore
.env
__pycache__
*.pyc
.ipynb_checkpoints
notebooks/
evaluation/
data/raw/
data/embeddings/
venv/
*.ipynb
README.md
"""

with open(".dockerignore", "w") as f:
    f.write(dockerignore)

print(".dockerignore written.")

---
## Next Steps & Research Roadmap

This notebook provides a complete pipeline from data collection to containerised deployment.
Here is what to tackle next to turn this into a full MSc-quality research project:

### Immediate (Week 1-2)
- [ ] Fill in `.env` with real API keys and run the data collection
- [ ] Run EDA and verify data quality
- [ ] Generate embeddings with all three models
- [ ] Run the qualitative retrieval tests and document observations

### Short-term (Week 3-4)
- [ ] Complete the full evaluation on all users (not just the subset)
- [ ] Implement the cold-start vs warm split analysis (Phase 8.5)
- [ ] Add NDCG@K metric (requires relevance scores, not just binary)
- [ ] Run statistical significance tests on the method comparison
- [ ] Experiment with different `top_k_retrieve` values (ablation study)

### Medium-term (Week 5-8)
- [ ] Try fine-tuning an embedding model on your music domain data
- [ ] Implement a hybrid approach (RAG + CF for warm users, RAG-only for cold)
- [ ] Add user preference modelling to the RAG context
- [ ] Evaluate with real user studies (if ethics approval permits)
- [ ] Benchmark latency (FAISS retrieval + LLM inference time)

### Deployment & Production (Week 9-10)
- [ ] Build and test Docker image
- [ ] Set up CI/CD pipeline (GitHub Actions)
- [ ] Choose deployment target (AWS ECS, GCP Cloud Run, etc.)
- [ ] Add monitoring and logging (Prometheus + Grafana or similar)
- [ ] Load testing with locust or k6

### Research Writing
- [ ] Literature review: existing cold-start solutions, RAG in RecSys
- [ ] Methodology: justify each design decision (embedding models, LLM choice, etc.)
- [ ] Results: tables + charts for all metrics, split by cold/warm
- [ ] Discussion: why RAG helps (or doesn't) for cold-start
- [ ] Limitations: synthetic interactions, API cost, LLM hallucination risk